# IMDb Screenplay Task C: High-vs-Low Success Classifier

This notebook is the improved **Task C** version.

**Goal:** Instead of predicting exact IMDb rating, classify clearly low-rated vs clearly high-rated films from screenplay text.

Default task:

- **Low / weak:** `rating <= 6.5`
- **High / strong:** `rating >= 7.5`
- **Middle:** dropped from training/evaluation

This notebook focuses on:
- no train/test contamination
- one movie = one sample
- no BERT fine-tuning
- no matplotlib dependency
- fast K-fold cross-validation
- ablations to check whether signal comes from real text, formatting, names, or numeric artifacts


In [1]:
# ============================================================
# 1) IMPORTS
# ============================================================

from pathlib import Path
import re
import math
import gc
import json
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import joblib

from tqdm.auto import tqdm
from IPython.display import display

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

warnings.filterwarnings("ignore")

print("Imports OK")
print("numpy:", np.__version__)
print("pandas:", pd.__version__)


Imports OK
numpy: 1.26.4
pandas: 2.2.3


In [2]:
# ============================================================
# 2) CONFIG
# ============================================================

RANDOM_SEED = 42

# Main Task C thresholds.
LOW_THRESHOLD = 6.5
HIGH_THRESHOLD = 7.5

# Also run stricter sensitivity experiments later.
EXPERIMENT_THRESHOLDS = [
    (6.5, 7.5),
    (6.0, 7.5),
    (6.0, 8.0),
]

# Cross-validation.
N_FOLDS = 5

# Fast TF-IDF caps. If the notebook is slow, lower these.
MAX_WORD_FEATURES = 12000
MAX_CHAR_FEATURES = 12000
MIN_DF = 3

# Logistic regression.
LOGREG_C = 1.0

# Confidence bands for "uncertain/middle" interpretation.
CONFIDENCE_LOW = 0.35
CONFIDENCE_HIGH = 0.65

# Output files.
RESULTS_PATH = Path("task_c_improved_ablation_results.csv")
PREDICTIONS_PATH = Path("task_c_improved_predictions.csv")
FINAL_MODEL_PATH = Path("task_c_improved_final_model.joblib")

print("Config loaded.")


Config loaded.


In [3]:
# ============================================================
# 3) LOCATE REPO FILES
# ============================================================

def find_repo_root():
    """
    Works if Jupyter starts from repo root or notebooks/.
    """
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path(r"C:\Users\RiceC\IMDB\Predicting-Screenplay-Success-Using-Film-Structure"),
    ]

    for candidate in candidates:
        if (candidate / "movie_ratings_final.csv").exists() and (candidate / "screenplay_dataset").exists():
            return candidate

    raise FileNotFoundError(
        "Could not find repo root with movie_ratings_final.csv and screenplay_dataset/. "
        "Open this notebook from the repo root or notebooks/ folder."
    )

REPO_DIR = find_repo_root()
CSV_PATH = REPO_DIR / "movie_ratings_final.csv"
SCREENPLAY_DIR = REPO_DIR / "screenplay_dataset"

print("Repo:", REPO_DIR)
print("CSV:", CSV_PATH)
print("Screenplay dir:", SCREENPLAY_DIR)


Repo: c:\Users\RiceC\IMDB\Predicting-Screenplay-Success-Using-Film-Structure
CSV: c:\Users\RiceC\IMDB\Predicting-Screenplay-Success-Using-Film-Structure\movie_ratings_final.csv
Screenplay dir: c:\Users\RiceC\IMDB\Predicting-Screenplay-Success-Using-Film-Structure\screenplay_dataset


In [4]:
# ============================================================
# 4) TITLE / SCRIPT LOADING HELPERS
# ============================================================

def normalize_title(title):
    title = str(title).lower().strip()
    title = title.replace("&", " and ")
    title = re.sub(r"\([^)]*\)", " ", title)
    title = re.sub(r"[^a-z0-9]+", " ", title)
    title = re.sub(r"\s+", " ", title).strip()
    title = re.sub(r"^(the|a|an)\s+", "", title)
    return title

def canonicalize_script_stem(stem):
    """
    Normalizes screenplay filename stems:
      Script_Matrix, The -> matrix
      Matrix, The        -> matrix
      The Matrix         -> matrix
    """
    stem = re.sub(r"^Script_", "", str(stem), flags=re.IGNORECASE).strip()
    stem = re.sub(r",\s*(The|A|An|L'|El|La|Le)$", "", stem, flags=re.IGNORECASE).strip()
    return normalize_title(stem)

def build_script_index(screenplay_dir):
    index = {}

    for path in screenplay_dir.rglob("*.txt"):
        norm = canonicalize_script_stem(path.stem)

        if norm and norm not in index:
            index[norm] = path

    return index

def clean_raw_text(text):
    if pd.isna(text):
        return ""

    text = str(text)
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\t+", " ", text)
    text = re.sub(r"[ ]{2,}", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def read_script(path):
    return clean_raw_text(path.read_text(encoding="utf-8", errors="ignore"))

script_index = build_script_index(SCREENPLAY_DIR)
print("Indexed .txt scripts:", len(script_index))
print("Sample scripts:")
for k, v in list(script_index.items())[:5]:
    print(" ", k, "->", v.name)


Indexed .txt scripts: 1088
Sample scripts:
  10 things i hate about you -> Script_10 Things I Hate About You.txt
  12 and holding -> Script_12 and Holding.txt
  12 monkeys -> Script_12 Monkeys.txt
  12 years a slave -> Script_12 Years a Slave.txt
  12 -> Script_12.txt


In [5]:
# ============================================================
# 5) LOAD RATINGS CSV AND MATCH TO LOCAL SCREENPLAYS
# ============================================================

def find_col(df, possible_names):
    lower_map = {str(col).lower().strip(): col for col in df.columns}

    for name in possible_names:
        key = name.lower().strip()
        if key in lower_map:
            return lower_map[key]

    return None

ratings_df = pd.read_csv(CSV_PATH)
print("CSV columns:", list(ratings_df.columns))
display(ratings_df.head())

title_col = find_col(ratings_df, ["original_title", "title", "movie_title", "name"])
rating_col = find_col(ratings_df, ["rating", "imdb_rating", "IMDb Rating", "averageRating"])

if title_col is None:
    raise ValueError(f"Could not find title column. Columns: {list(ratings_df.columns)}")
if rating_col is None:
    raise ValueError(f"Could not find rating column. Columns: {list(ratings_df.columns)}")

ratings_df[rating_col] = pd.to_numeric(ratings_df[rating_col], errors="coerce")
ratings_df = ratings_df.dropna(subset=[title_col, rating_col]).copy()

rows = []
missing = []

for _, row in tqdm(ratings_df.iterrows(), total=len(ratings_df), desc="Matching scripts"):
    title = str(row[title_col])
    norm = normalize_title(title)
    script_path = script_index.get(norm)

    if script_path is None:
        missing.append(title)
        continue

    text = read_script(script_path)
    word_count = len(text.split())

    if word_count < 500:
        continue

    rows.append({
        "title": title,
        "norm_title": norm,
        "rating": float(row[rating_col]),
        "script_path": str(script_path),
        "script_text": text,
        "word_count": word_count,
    })

matched_df = pd.DataFrame(rows)

print("Rows in ratings CSV:", len(ratings_df))
print("Usable matched scripts:", len(matched_df))
print("Missing title matches:", len(missing))
print("Rating range:", matched_df["rating"].min(), "to", matched_df["rating"].max())
display(matched_df[["title", "rating", "word_count", "script_path"]].head())


CSV columns: ['original_title', 'rating', 'is_successful']


,original_title,rating,is_successful
0,10 Things I Hate About You,7.4,1
1,12,7.5,1
2,12 and Holding,7.4,1
3,12 Monkeys,8.0,1
4,12 Years a Slave,8.1,1


Matching scripts:   0%|          | 0/1076 [00:00<?, ?it/s]

Rows in ratings CSV: 1076
Usable matched scripts: 1036
Missing title matches: 40
Rating range: 2.4 to 9.3


,title,rating,word_count,script_path
0,10 Things I Hate About You,7.4,18588,c:\Users\RiceC\IMDB\Predicting-Screenplay-Succ...
1,12,7.5,13190,c:\Users\RiceC\IMDB\Predicting-Screenplay-Succ...
2,12 and Holding,7.4,18362,c:\Users\RiceC\IMDB\Predicting-Screenplay-Succ...
3,12 Monkeys,8.0,30879,c:\Users\RiceC\IMDB\Predicting-Screenplay-Succ...
4,12 Years a Slave,8.1,30386,c:\Users\RiceC\IMDB\Predicting-Screenplay-Succ...


In [6]:
# ============================================================
# 6) CLEAN LINE PARSING + NUMERIC FEATURES
# ============================================================

def repair_screenplay_newlines(text):
    """
    Some scripts are stored as huge lines. This adds newlines before obvious
    screenplay boundaries so line-based features are less fake.
    """
    text = clean_raw_text(text)

    # Add line breaks before scene headings.
    text = re.sub(r"\s+(?=(INT\.|EXT\.|INT/EXT\.|I/E\.)\s)", "\n", text, flags=re.IGNORECASE)

    # Add line breaks before common transitions.
    text = re.sub(r"\s+(?=(CUT TO:|FADE IN:|FADE OUT:|DISSOLVE TO:|SMASH CUT:))", "\n", text, flags=re.IGNORECASE)

    # Normalize repeated spaces/newlines.
    text = re.sub(r"[ ]{2,}", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

def mask_possible_character_names(text):
    """
    Rough artifact-control variant:
    replaces obvious all-caps screenplay character cues with CHARACTER_NAME.
    This is not perfect, but it helps test whether the model relies on names.
    """
    text = repair_screenplay_newlines(text)
    lines = text.splitlines()
    masked_lines = []

    for line in lines:
        stripped = line.strip()

        letters = re.sub(r"[^A-Za-z]", "", stripped)
        words = stripped.split()

        # All-caps short lines are often character cues.
        if (
            1 <= len(words) <= 4
            and len(letters) >= 3
            and letters.isupper()
            and not re.match(r"^(INT\.|EXT\.|CUT TO:|FADE|DISSOLVE)", stripped, flags=re.IGNORECASE)
        ):
            masked_lines.append("CHARACTER_NAME")
        else:
            masked_lines.append(line)

    return "\n".join(masked_lines)

POSITIVE_WORDS = {
    "love", "hope", "happy", "smile", "laugh", "dream", "trust", "friend",
    "beautiful", "free", "win", "safe", "home", "believe", "joy"
}

NEGATIVE_WORDS = {
    "kill", "death", "dead", "blood", "hate", "fear", "angry", "cry",
    "pain", "gun", "knife", "murder", "scream", "alone", "lost", "war"
}

def extract_numeric_features(text):
    text = repair_screenplay_newlines(text)
    upper_text = text.upper()

    words = re.findall(r"[A-Za-z']+", text.lower())
    word_count = len(words)

    lines = [line.strip() for line in text.splitlines() if line.strip()]
    line_count = len(lines)

    scene_count = len(re.findall(r"\b(INT\.|EXT\.|INT/EXT\.|I/E\.)", upper_text))

    uppercase_lines = 0
    short_lines = 0
    long_lines = 0
    dialogue_like_lines = 0

    for line in lines:
        line_words = line.split()
        letters = re.sub(r"[^A-Za-z]", "", line)

        if len(line_words) <= 4:
            short_lines += 1

        if len(line_words) >= 20:
            long_lines += 1

        if len(letters) >= 3 and letters.isupper() and len(line_words) <= 4:
            uppercase_lines += 1
            dialogue_like_lines += 1

    avg_line_words = word_count / max(1, line_count)
    scene_density = scene_count / max(1, word_count / 1000)

    pos_count = sum(1 for w in words if w in POSITIVE_WORDS)
    neg_count = sum(1 for w in words if w in NEGATIVE_WORDS)

    return {
        "log_word_count": np.log1p(word_count),
        "log_line_count": np.log1p(line_count),
        "log_scene_count": np.log1p(scene_count),
        "avg_line_words": avg_line_words,
        "scene_density_per_1k_words": scene_density,
        "uppercase_line_ratio": uppercase_lines / max(1, line_count),
        "short_line_ratio": short_lines / max(1, line_count),
        "long_line_ratio": long_lines / max(1, line_count),
        "dialogue_cue_ratio": dialogue_like_lines / max(1, line_count),
        "question_density": text.count("?") / max(1, word_count),
        "exclamation_density": text.count("!") / max(1, word_count),
        "positive_word_density": pos_count / max(1, word_count),
        "negative_word_density": neg_count / max(1, word_count),
        "neg_minus_pos_density": (neg_count - pos_count) / max(1, word_count),
    }

feature_rows = []

for text in tqdm(matched_df["script_text"], desc="Extracting numeric features"):
    feature_rows.append(extract_numeric_features(text))

features_df = pd.DataFrame(feature_rows)

model_base_df = pd.concat([
    matched_df.reset_index(drop=True),
    features_df.reset_index(drop=True),
], axis=1)

model_base_df["script_text_repaired"] = model_base_df["script_text"].apply(repair_screenplay_newlines)
model_base_df["script_text_masked"] = model_base_df["script_text"].apply(mask_possible_character_names)

NUMERIC_FEATURES = list(features_df.columns)

print("Numeric features:")
print(NUMERIC_FEATURES)

print("\nFeature sanity check:")
display(model_base_df[["title", "rating", "word_count", "log_line_count", "avg_line_words", "log_scene_count", "scene_density_per_1k_words"]].head(10))

print("\nFeature summary:")
display(model_base_df[NUMERIC_FEATURES].describe().T)


Extracting numeric features:   0%|          | 0/1036 [00:00<?, ?it/s]

Numeric features:
['log_word_count', 'log_line_count', 'log_scene_count', 'avg_line_words', 'scene_density_per_1k_words', 'uppercase_line_ratio', 'short_line_ratio', 'long_line_ratio', 'dialogue_cue_ratio', 'question_density', 'exclamation_density', 'positive_word_density', 'negative_word_density', 'neg_minus_pos_density']

Feature sanity check:


,title,rating,word_count,log_line_count,avg_line_words,log_scene_count,scene_density_per_1k_words
0,10 Things I Hate About You,7.4,18588,4.521789,202.483516,4.499810,4.830131
1,12,7.5,13190,0.693147,13370.000000,0.000000,0.000000
2,12 and Holding,7.4,18362,5.170484,102.565714,5.093750,9.025572
3,12 Monkeys,8.0,30879,5.176150,176.289773,5.170484,5.640249
4,12 Years a Slave,8.1,30386,5.049856,190.541935,5.036953,5.180470
5,127 Hours,7.5,18341,6.129050,39.613537,5.389072,12.015653
6,1492: Conquest of Paradise,6.4,22913,5.379897,104.564815,5.056246,6.906933
7,15 Minutes,6.1,25209,4.317488,336.675676,4.276666,2.849803
8,17 Again,6.4,18016,4.859812,140.359375,4.770685,6.512301
9,2001: A Space Odyssey,8.3,14281,0.693147,15214.000000,0.000000,0.000000



Feature summary:


,count,mean,std,min,25%,50%,75%,max
log_word_count,1036.0,10.014712,0.288040,7.352441,9.904100,10.044900,10.177752,11.124775
log_line_count,1036.0,4.440665,1.524399,0.693147,4.234107,4.919981,5.267858,8.571303
log_scene_count,1036.0,4.327418,1.581911,0.000000,4.442651,4.875197,5.141664,6.448889
avg_line_words,1036.0,1992.720296,5882.053159,4.415323,121.808509,164.392245,337.983158,67830.000000
scene_density_per_1k_words,1036.0,5.555209,3.180867,0.000000,3.830443,5.690253,7.406922,21.162424
uppercase_line_ratio,1036.0,0.077264,0.120742,0.000000,0.000000,0.017417,0.099189,0.500000
short_line_ratio,1036.0,0.085555,0.136602,0.000000,0.000000,0.019893,0.104813,0.661029
long_line_ratio,1036.0,0.843057,0.183016,0.000000,0.798416,0.898642,0.963076,1.000000
dialogue_cue_ratio,1036.0,0.077264,0.120742,0.000000,0.000000,0.017417,0.099189,0.500000
question_density,1036.0,0.012900,0.004979,0.001032,0.009409,0.012155,0.015765,0.041672


In [7]:
# ============================================================
# 7) BUILD TASK C DATASET
# ============================================================

def build_task_c_df(base_df, low_threshold=6.5, high_threshold=7.5):
    df = base_df.copy()

    df = df[(df["rating"] <= low_threshold) | (df["rating"] >= high_threshold)].copy()
    df["task_c_label"] = (df["rating"] >= high_threshold).astype(int)

    # Deduplicate normalized titles BEFORE CV. Keep first.
    df = df.sort_values(["norm_title", "rating"]).drop_duplicates(subset=["norm_title"], keep="first")
    df = df.reset_index(drop=True)

    return df

model_df = build_task_c_df(model_base_df, LOW_THRESHOLD, HIGH_THRESHOLD)

print(f"Task C thresholds: low <= {LOW_THRESHOLD}, high >= {HIGH_THRESHOLD}")
print("Task C usable movies:", len(model_df))
print(model_df["task_c_label"].value_counts().rename(index={0: "low", 1: "high"}))

assert len(model_df["norm_title"]) == len(set(model_df["norm_title"])), "Duplicate normalized titles remain."

display(model_df[["title", "rating", "task_c_label", "word_count"]].head(10))


Task C thresholds: low <= 6.5, high >= 7.5
Task C usable movies: 623
task_c_label
low     321
high    302
Name: count, dtype: int64


,title,rating,task_c_label,word_count
0,12,7.5,1,13190
1,12 Monkeys,8.0,1,30879
2,12 Years a Slave,8.1,1,30386
3,127 Hours,7.5,1,18341
4,1492: Conquest of Paradise,6.4,0,22913
5,15 Minutes,6.1,0,25209
6,17 Again,6.4,0,18016
7,2001: A Space Odyssey,8.3,1,14281
8,2012,5.9,0,25857
9,25th Hour,7.6,1,21942


In [8]:
# ============================================================
# 8) MODEL PIPELINES
# ============================================================

def make_pipeline_variant(variant_name, text_col="script_text_repaired"):
    """
    Ablation variants:
      word_only
      char_only
      numeric_only
      word_char
      full
      masked_full
    """
    transformers = []

    if variant_name in {"word_only", "word_char", "full", "masked_full"}:
        transformers.append((
            "word_tfidf",
            TfidfVectorizer(
                lowercase=True,
                strip_accents="unicode",
                stop_words="english",
                ngram_range=(1, 2),
                min_df=MIN_DF,
                max_df=0.90,
                max_features=MAX_WORD_FEATURES,
                sublinear_tf=True,
                dtype=np.float32,
            ),
            text_col,
        ))

    if variant_name in {"char_only", "word_char", "full", "masked_full"}:
        transformers.append((
            "char_tfidf",
            TfidfVectorizer(
                lowercase=True,
                analyzer="char_wb",
                ngram_range=(3, 5),
                min_df=MIN_DF,
                max_df=0.95,
                max_features=MAX_CHAR_FEATURES,
                sublinear_tf=True,
                dtype=np.float32,
            ),
            text_col,
        ))

    if variant_name in {"numeric_only", "full", "masked_full"}:
        transformers.append((
            "numeric",
            StandardScaler(),
            NUMERIC_FEATURES,
        ))

    if not transformers:
        raise ValueError(f"Unknown variant: {variant_name}")

    preprocessor = ColumnTransformer(
        transformers=transformers,
        sparse_threshold=0.95,
    )

    clf = LogisticRegression(
        C=LOGREG_C,
        class_weight="balanced",
        solver="liblinear",
        max_iter=1500,
        random_state=RANDOM_SEED,
    )

    return Pipeline([
        ("features", preprocessor),
        ("clf", clf),
    ])

def safe_roc_auc(y_true, probs):
    if len(np.unique(y_true)) < 2:
        return np.nan
    return roc_auc_score(y_true, probs)

def metrics_at_threshold(y_true, probs, threshold=0.5):
    pred = (probs >= threshold).astype(int)

    cm = confusion_matrix(y_true, pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    return {
        "accuracy": accuracy_score(y_true, pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, pred),
        "macro_f1": f1_score(y_true, pred, average="macro", zero_division=0),
        "roc_auc": safe_roc_auc(y_true, probs),
        "low_recall": tn / max(1, tn + fp),
        "high_recall": tp / max(1, tp + fn),
        "pred_low_count": int(np.sum(pred == 0)),
        "pred_high_count": int(np.sum(pred == 1)),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }

def evaluate_variant_cv(df, variant_name, text_col="script_text_repaired", n_folds=5):
    X = df[[text_col] + NUMERIC_FEATURES].copy()
    y = df["task_c_label"].to_numpy()
    titles = df["title"].astype(str).to_numpy()
    ratings = df["rating"].astype(float).to_numpy()
    norm_titles = df["norm_title"].astype(str).to_numpy()

    assert len(norm_titles) == len(set(norm_titles)), "Duplicate normalized titles remain."

    min_class_count = int(np.bincount(y).min())
    folds = min(n_folds, min_class_count)

    cv = StratifiedKFold(
        n_splits=folds,
        shuffle=True,
        random_state=RANDOM_SEED,
    )

    rows = []
    pred_rows = []
    last_model = None

    print(f"\nEvaluating variant: {variant_name}")
    print(f"Rows={len(y)} | low={np.sum(y == 0)} high={np.sum(y == 1)} | folds={folds}")

    for fold_id, (train_idx, test_idx) in enumerate(tqdm(list(cv.split(X, y)), desc=f"{variant_name} folds"), start=1):
        X_train = X.iloc[train_idx].copy()
        X_test = X.iloc[test_idx].copy()
        y_train = y[train_idx]
        y_test = y[test_idx]

        train_titles = set(norm_titles[train_idx])
        test_titles = set(norm_titles[test_idx])
        assert train_titles.isdisjoint(test_titles), "Train/test title contamination detected."

        model = make_pipeline_variant(variant_name, text_col=text_col)
        model.fit(X_train, y_train)

        probs = model.predict_proba(X_test)[:, 1]
        threshold = 0.5

        row = {
            "variant": variant_name,
            "fold": fold_id,
            "threshold": threshold,
            "n_train": len(train_idx),
            "n_test": len(test_idx),
            "train_low": int(np.sum(y_train == 0)),
            "train_high": int(np.sum(y_train == 1)),
            "test_low": int(np.sum(y_test == 0)),
            "test_high": int(np.sum(y_test == 1)),
        }

        row.update(metrics_at_threshold(y_test, probs, threshold))

        majority = DummyClassifier(strategy="most_frequent")
        majority.fit(X_train, y_train)
        majority_pred = majority.predict(X_test)

        row["majority_accuracy"] = accuracy_score(y_test, majority_pred)
        row["majority_balanced_accuracy"] = balanced_accuracy_score(y_test, majority_pred)
        row["majority_macro_f1"] = f1_score(y_test, majority_pred, average="macro", zero_division=0)

        # Confidence bucket metrics.
        low_conf = probs <= CONFIDENCE_LOW
        high_conf = probs >= CONFIDENCE_HIGH
        confident = low_conf | high_conf

        row["confident_coverage"] = float(np.mean(confident))
        if np.sum(confident) >= 5:
            conf_pred = (probs[confident] >= 0.5).astype(int)
            row["confident_accuracy"] = accuracy_score(y_test[confident], conf_pred)
            if len(np.unique(y_test[confident])) == 2:
                row["confident_balanced_accuracy"] = balanced_accuracy_score(y_test[confident], conf_pred)
            else:
                row["confident_balanced_accuracy"] = np.nan
        else:
            row["confident_accuracy"] = np.nan
            row["confident_balanced_accuracy"] = np.nan

        rows.append(row)

        fold_pred_df = pd.DataFrame({
            "variant": variant_name,
            "fold": fold_id,
            "title": titles[test_idx],
            "rating": ratings[test_idx],
            "actual_label": y_test,
            "pred_prob_high": probs,
            "pred_label": (probs >= threshold).astype(int),
            "confidence_bucket": np.where(probs <= CONFIDENCE_LOW, "low_confident",
                                np.where(probs >= CONFIDENCE_HIGH, "high_confident", "uncertain")),
        })
        pred_rows.append(fold_pred_df)

        last_model = model

        print(
            f"  fold {fold_id}/{folds}: "
            f"bal_acc={row['balanced_accuracy']:.3f}, "
            f"auc={row['roc_auc']:.3f}, "
            f"low_rec={row['low_recall']:.3f}, "
            f"high_rec={row['high_recall']:.3f}, "
            f"conf_cov={row['confident_coverage']:.2f}"
        )

        del X_train, X_test, y_train, y_test, model, majority
        gc.collect()

    return pd.DataFrame(rows), pd.concat(pred_rows, ignore_index=True), last_model

def summarize_results(results_df):
    metric_cols = [
        "accuracy",
        "balanced_accuracy",
        "macro_f1",
        "roc_auc",
        "low_recall",
        "high_recall",
        "confident_coverage",
        "confident_accuracy",
        "confident_balanced_accuracy",
        "majority_accuracy",
        "majority_balanced_accuracy",
        "majority_macro_f1",
    ]

    summary = (
        results_df
        .groupby("variant")[metric_cols]
        .agg(["mean", "std"])
        .reset_index()
    )

    # Flatten columns.
    summary.columns = [
        col[0] if col[1] == "" else f"{col[0]}_{col[1]}"
        for col in summary.columns
    ]

    return summary


In [9]:
# ============================================================
# 9) RUN MAIN ABLATIONS
# ============================================================

# These are the important sanity checks:
# - word_only: real word/phrase text signal
# - char_only: formatting/name/source artifacts can show up here
# - numeric_only: checks whether line/length/scene features alone drive result
# - word_char: text-only without numeric features
# - full: all features
# - masked_full: same as full, but character cue names masked

VARIANTS = [
    ("word_only", "script_text_repaired"),
    ("char_only", "script_text_repaired"),
    ("numeric_only", "script_text_repaired"),
    ("word_char", "script_text_repaired"),
    ("full", "script_text_repaired"),
    ("masked_full", "script_text_masked"),
]

all_result_dfs = []
all_prediction_dfs = []
trained_models = {}

for variant, text_col in VARIANTS:
    result_df, pred_df, last_model = evaluate_variant_cv(
        model_df,
        variant_name=variant,
        text_col=text_col,
        n_folds=N_FOLDS,
    )

    all_result_dfs.append(result_df)
    all_prediction_dfs.append(pred_df)
    trained_models[variant] = last_model

results_df = pd.concat(all_result_dfs, ignore_index=True)
predictions_df = pd.concat(all_prediction_dfs, ignore_index=True)

summary_df = summarize_results(results_df)

print("\nAblation summary:")
display(summary_df.sort_values("balanced_accuracy_mean", ascending=False))

results_df.to_csv(RESULTS_PATH, index=False)
predictions_df.to_csv(PREDICTIONS_PATH, index=False)

print("\nSaved:")
print(" ", RESULTS_PATH)
print(" ", PREDICTIONS_PATH)



Evaluating variant: word_only
Rows=623 | low=321 high=302 | folds=5


word_only folds:   0%|          | 0/5 [00:00<?, ?it/s]

  fold 1/5: bal_acc=0.640, auc=0.744, low_rec=0.625, high_rec=0.656, conf_cov=0.14
  fold 2/5: bal_acc=0.727, auc=0.804, low_rec=0.781, high_rec=0.672, conf_cov=0.14
  fold 3/5: bal_acc=0.726, auc=0.751, low_rec=0.769, high_rec=0.683, conf_cov=0.16
  fold 4/5: bal_acc=0.718, auc=0.775, low_rec=0.703, high_rec=0.733, conf_cov=0.15
  fold 5/5: bal_acc=0.658, auc=0.744, low_rec=0.750, high_rec=0.567, conf_cov=0.17

Evaluating variant: char_only
Rows=623 | low=321 high=302 | folds=5


char_only folds:   0%|          | 0/5 [00:00<?, ?it/s]

  fold 1/5: bal_acc=0.647, auc=0.741, low_rec=0.688, high_rec=0.607, conf_cov=0.09
  fold 2/5: bal_acc=0.703, auc=0.785, low_rec=0.750, high_rec=0.656, conf_cov=0.16
  fold 3/5: bal_acc=0.718, auc=0.774, low_rec=0.769, high_rec=0.667, conf_cov=0.15
  fold 4/5: bal_acc=0.702, auc=0.770, low_rec=0.703, high_rec=0.700, conf_cov=0.15
  fold 5/5: bal_acc=0.707, auc=0.742, low_rec=0.781, high_rec=0.633, conf_cov=0.16

Evaluating variant: numeric_only
Rows=623 | low=321 high=302 | folds=5


...

  fold 1/5: bal_acc=0.679, auc=0.766, low_rec=0.703, high_rec=0.656, conf_cov=0.39
  fold 2/5: bal_acc=0.735, auc=0.800, low_rec=0.766, high_rec=0.705, conf_cov=0.48
  fold 3/5: bal_acc=0.702, auc=0.741, low_rec=0.754, high_rec=0.650, conf_cov=0.45
  fold 4/5: bal_acc=0.684, auc=0.722, low_rec=0.734, high_rec=0.633, conf_cov=0.42
  fold 5/5: bal_acc=0.667, auc=0.728, low_rec=0.750, high_rec=0.583, conf_cov=0.48

Evaluating variant: masked_full
Rows=623 | low=321 high=302 | folds=5


masked_full folds:   0%|          | 0/5 [00:00<?, ?it/s]

  fold 1/5: bal_acc=0.687, auc=0.763, low_rec=0.719, high_rec=0.656, conf_cov=0.40
  fold 2/5: bal_acc=0.752, auc=0.802, low_rec=0.766, high_rec=0.738, conf_cov=0.48
  fold 3/5: bal_acc=0.702, auc=0.743, low_rec=0.754, high_rec=0.650, conf_cov=0.45
  fold 4/5: bal_acc=0.684, auc=0.720, low_rec=0.734, high_rec=0.633, conf_cov=0.42
  fold 5/5: bal_acc=0.667, auc=0.731, low_rec=0.750, high_rec=0.583, conf_cov=0.47

Ablation summary:


,variant,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std,macro_f1_mean,macro_f1_std,roc_auc_mean,roc_auc_std,low_recall_mean,...,confident_accuracy_mean,confident_accuracy_std,confident_balanced_accuracy_mean,confident_balanced_accuracy_std,majority_accuracy_mean,majority_accuracy_std,majority_balanced_accuracy_mean,majority_balanced_accuracy_std,majority_macro_f1_mean,majority_macro_f1_std
4,word_char,0.717484,0.025060,0.716558,0.024892,0.716345,0.024933,0.775037,0.019830,0.747596,...,0.843551,0.035895,0.844114,0.036714,0.515252,0.003363,0.5,0.0,0.340041,0.001464
2,masked_full,0.699768,0.031676,0.698271,0.032375,0.698123,0.032657,0.751939,0.032340,0.744519,...,0.789920,0.062712,0.788407,0.064206,0.515252,0.003363,0.5,0.0,0.340041,0.001464
0,char_only,0.696658,0.028109,0.695340,0.027764,0.695249,0.027823,0.762183,0.020034,0.738221,...,0.858017,0.042559,0.843207,0.041689,0.515252,0.003363,0.5,0.0,0.340041,0.001464
5,word_only,0.695006,0.041402,0.693981,0.041371,0.693561,0.041762,0.763638,0.025993,0.725721,...,0.893367,0.035949,0.892435,0.038662,0.515252,0.003363,0.5,0.0,0.340041,0.001464
1,full,0.694968,0.026150,0.693429,0.026584,0.693294,0.026891,0.751373,0.031960,0.741394,...,0.787271,0.059882,0.785866,0.060698,0.515252,0.003363,0.5,0.0,0.340041,0.001464
3,numeric_only,0.601858,0.036601,0.600230,0.037186,0.599386,0.038110,0.644675,0.035996,0.654231,...,0.691704,0.055188,0.695390,0.049124,0.515252,0.003363,0.5,0.0,0.340041,0.001464



Saved:
  task_c_improved_ablation_results.csv
  task_c_improved_predictions.csv


In [11]:
# ============================================================
# 10) STRICTER THRESHOLD SENSITIVITY EXPERIMENTS
# Same model, same 5 folds, just safer memory cleanup.
# ============================================================

import gc

threshold_experiment_rows = []

for low_t, high_t in EXPERIMENT_THRESHOLDS:
    gc.collect()

    temp_df = build_task_c_df(model_base_df, low_t, high_t)

    print("\n" + "=" * 80)
    print(f"Experiment: low <= {low_t}, high >= {high_t}")
    print("Rows:", len(temp_df))
    print(temp_df["task_c_label"].value_counts().rename(index={0: "low", 1: "high"}))

    if len(temp_df) < 50 or temp_df["task_c_label"].nunique() < 2:
        print("Skipping: too few examples.")
        del temp_df
        gc.collect()
        continue

    result_df, pred_df, _ = evaluate_variant_cv(
        temp_df,
        variant_name="full",
        text_col="script_text_repaired",
        n_folds=N_FOLDS,   # KEEPING 5 FOLDS
    )

    result_df["low_threshold"] = low_t
    result_df["high_threshold"] = high_t

    threshold_experiment_rows.append(result_df.copy())

    # Important: predictions are not needed for threshold summary.
    # Keeping them around wastes memory.
    del pred_df
    del result_df
    del temp_df
    gc.collect()

if threshold_experiment_rows:
    threshold_results_df = pd.concat(threshold_experiment_rows, ignore_index=True)

    threshold_summary = (
        threshold_results_df
        .groupby(["low_threshold", "high_threshold"])
        [["accuracy", "balanced_accuracy", "macro_f1", "roc_auc", "low_recall", "high_recall", "majority_balanced_accuracy"]]
        .agg(["mean", "std"])
        .reset_index()
    )

    threshold_summary.columns = [
        col[0] if col[1] == "" else f"{col[0]}_{col[1]}"
        for col in threshold_summary.columns
    ]

    print("\nThreshold sensitivity summary:")
    display(threshold_summary.sort_values("balanced_accuracy_mean", ascending=False))

    threshold_results_df.to_csv("task_c_threshold_sensitivity_results.csv", index=False)
else:
    print("No threshold experiments were run.")


Experiment: low <= 6.5, high >= 7.5
Rows: 623
task_c_label
low     321
high    302
Name: count, dtype: int64

Evaluating variant: full
Rows=623 | low=321 high=302 | folds=5


full folds:   0%|          | 0/5 [00:00<?, ?it/s]

  fold 1/5: bal_acc=0.679, auc=0.766, low_rec=0.703, high_rec=0.656, conf_cov=0.39
  fold 2/5: bal_acc=0.735, auc=0.800, low_rec=0.766, high_rec=0.705, conf_cov=0.48
  fold 3/5: bal_acc=0.702, auc=0.741, low_rec=0.754, high_rec=0.650, conf_cov=0.45
  fold 4/5: bal_acc=0.684, auc=0.722, low_rec=0.734, high_rec=0.633, conf_cov=0.42
  fold 5/5: bal_acc=0.667, auc=0.728, low_rec=0.750, high_rec=0.583, conf_cov=0.48

Experiment: low <= 6.0, high >= 7.5
Rows: 466
task_c_label
high    302
low     164
Name: count, dtype: int64

Evaluating variant: full
Rows=466 | low=164 high=302 | folds=5


full folds:   0%|          | 0/5 [00:00<?, ?it/s]

  fold 1/5: bal_acc=0.732, auc=0.794, low_rec=0.727, high_rec=0.738, conf_cov=0.46
  fold 2/5: bal_acc=0.760, auc=0.856, low_rec=0.750, high_rec=0.770, conf_cov=0.51
  fold 3/5: bal_acc=0.692, auc=0.750, low_rec=0.667, high_rec=0.717, conf_cov=0.57
  fold 4/5: bal_acc=0.652, auc=0.724, low_rec=0.636, high_rec=0.667, conf_cov=0.46
  fold 5/5: bal_acc=0.652, auc=0.705, low_rec=0.636, high_rec=0.667, conf_cov=0.56

Experiment: low <= 6.0, high >= 8.0
Rows: 277
task_c_label
low     164
high    113
Name: count, dtype: int64

Evaluating variant: full
Rows=277 | low=164 high=113 | folds=5


full folds:   0%|          | 0/5 [00:00<?, ?it/s]

  fold 1/5: bal_acc=0.683, auc=0.776, low_rec=0.758, high_rec=0.609, conf_cov=0.50
  fold 2/5: bal_acc=0.679, auc=0.823, low_rec=0.879, high_rec=0.478, conf_cov=0.52
  fold 3/5: bal_acc=0.673, auc=0.696, low_rec=0.781, high_rec=0.565, conf_cov=0.56
  fold 4/5: bal_acc=0.636, auc=0.691, low_rec=0.727, high_rec=0.545, conf_cov=0.60
  fold 5/5: bal_acc=0.720, auc=0.738, low_rec=0.848, high_rec=0.591, conf_cov=0.55

Threshold sensitivity summary:


,low_threshold,high_threshold,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std,macro_f1_mean,macro_f1_std,roc_auc_mean,roc_auc_std,low_recall_mean,low_recall_std,high_recall_mean,high_recall_std,majority_balanced_accuracy_mean,majority_balanced_accuracy_std
0,6.0,7.5,0.701647,0.047588,0.697486,0.048540,0.686541,0.047647,0.765733,0.060683,0.683333,0.052596,0.711639,0.045314,0.5,0.0
2,6.5,7.5,0.694968,0.026150,0.693429,0.026584,0.693294,0.026891,0.751373,0.031960,0.741394,0.024135,0.645464,0.043784,0.5,0.0
1,6.0,8.0,0.700325,0.033284,0.678191,0.029671,0.680670,0.031178,0.744975,0.055821,0.798674,0.063226,0.557708,0.050548,0.5,0.0


In [12]:
# ============================================================
# 11) TRAIN FINAL MODEL ON ALL TASK C DATA + EXTRACT TERMS
# ============================================================

BEST_VARIANT = "full"
BEST_TEXT_COL = "script_text_repaired"

X_all = model_df[[BEST_TEXT_COL] + NUMERIC_FEATURES].copy()
y_all = model_df["task_c_label"].to_numpy()

final_model = make_pipeline_variant(BEST_VARIANT, text_col=BEST_TEXT_COL)
final_model.fit(X_all, y_all)

joblib.dump(final_model, FINAL_MODEL_PATH)
print("Saved final model:", FINAL_MODEL_PATH)

# Extract coefficients for interpretability.
def get_top_coefficients(pipeline, top_n=30):
    pre = pipeline.named_steps["features"]
    clf = pipeline.named_steps["clf"]

    feature_names = []

    for name, transformer, cols in pre.transformers_:
        if name == "remainder" and transformer == "drop":
            continue

        if name == "numeric":
            feature_names.extend([f"numeric: {x}" for x in cols])
        elif hasattr(transformer, "get_feature_names_out"):
            raw_names = transformer.get_feature_names_out()
            feature_names.extend([f"{name}: {x}" for x in raw_names])
        else:
            try:
                raw_names = transformer.get_feature_names_out()
                feature_names.extend([f"{name}: {x}" for x in raw_names])
            except Exception:
                feature_names.extend([f"{name}: {x}" for x in cols])

    coefs = clf.coef_[0]
    feature_names = np.array(feature_names)

    n = min(top_n, len(coefs))

    high_idx = np.argsort(coefs)[-n:][::-1]
    low_idx = np.argsort(coefs)[:n]

    high_df = pd.DataFrame({
        "feature": feature_names[high_idx],
        "coef": coefs[high_idx],
        "direction": "high_rating_associated",
    })

    low_df = pd.DataFrame({
        "feature": feature_names[low_idx],
        "coef": coefs[low_idx],
        "direction": "low_rating_associated",
    })

    return high_df, low_df

high_terms_df, low_terms_df = get_top_coefficients(final_model, top_n=40)

print("\nTop high-rating associated features:")
display(high_terms_df)

print("\nTop low-rating associated features:")
display(low_terms_df)

high_terms_df.to_csv("task_c_top_high_features.csv", index=False)
low_terms_df.to_csv("task_c_top_low_features.csv", index=False)


Saved final model: task_c_improved_final_model.joblib

Top high-rating associated features:


,feature,coef,direction
0,char_tfidf: -----,0.526111,high_rating_associated
1,char_tfidf: ----,0.495844,high_rating_associated
2,word_tfidf: melvin,0.470329,high_rating_associated
3,numeric: log_word_count,0.386531,high_rating_associated
4,char_tfidf: ---,0.377010,high_rating_associated
5,word_tfidf: aunt,0.333332,high_rating_associated
6,char_tfidf: elvin,0.300536,high_rating_associated
7,word_tfidf: mr,0.292576,high_rating_associated
8,word_tfidf: sir,0.288220,high_rating_associated
9,word_tfidf: old man,0.284655,high_rating_associated



Top low-rating associated features:


,feature,coef,direction
0,numeric: short_line_ratio,-0.569430,low_rating_associated
1,numeric: long_line_ratio,-0.491976,low_rating_associated
2,numeric: log_scene_count,-0.400421,low_rating_associated
3,word_tfidf: cable,-0.356926,low_rating_associated
4,word_tfidf: amy,-0.349289,low_rating_associated
5,word_tfidf: larry,-0.333391,low_rating_associated
6,word_tfidf: cell,-0.326575,low_rating_associated
7,numeric: negative_word_density,-0.324088,low_rating_associated
8,word_tfidf: cell phone,-0.318235,low_rating_associated
9,char_tfidf: juli,-0.313537,low_rating_associated


In [13]:
# ============================================================
# 12) HOW TO INTERPRET THE RESULT FOR THE PAPER
# ============================================================

best_summary = summary_df.sort_values("balanced_accuracy_mean", ascending=False).iloc[0]

print("Best ablation by balanced accuracy:")
display(best_summary.to_frame().T)

print("""
Paper-safe interpretation template:

We reframed screenplay success prediction as an extreme high-vs-low classification task.
Films rated <= 6.5 were treated as low reception, films rated >= 7.5 were treated as high reception,
and middle-rated films were excluded to reduce label noise.

Using movie-level 5-fold stratified cross-validation with TF-IDF and screenplay-style features,
the best model achieved the performance shown above compared to a majority baseline.

Important caveat:
This does NOT mean we can predict exact IMDb rating. It means screenplay text contains
some signal for separating clearly low-rated from clearly high-rated films. Ablation and
masked-name experiments should be reported to show whether the signal is robust or partly
driven by artifacts such as names, genre vocabulary, and formatting.
""")


Best ablation by balanced accuracy:


,variant,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std,macro_f1_mean,macro_f1_std,roc_auc_mean,roc_auc_std,low_recall_mean,...,confident_accuracy_mean,confident_accuracy_std,confident_balanced_accuracy_mean,confident_balanced_accuracy_std,majority_accuracy_mean,majority_accuracy_std,majority_balanced_accuracy_mean,majority_balanced_accuracy_std,majority_macro_f1_mean,majority_macro_f1_std
4,word_char,0.717484,0.02506,0.716558,0.024892,0.716345,0.024933,0.775037,0.01983,0.747596,...,0.843551,0.035895,0.844114,0.036714,0.515252,0.003363,0.5,0.0,0.340041,0.001464



Paper-safe interpretation template:

We reframed screenplay success prediction as an extreme high-vs-low classification task.
Films rated <= 6.5 were treated as low reception, films rated >= 7.5 were treated as high reception,
and middle-rated films were excluded to reduce label noise.

Using movie-level 5-fold stratified cross-validation with TF-IDF and screenplay-style features,
the best model achieved the performance shown above compared to a majority baseline.

Important caveat:
This does NOT mean we can predict exact IMDb rating. It means screenplay text contains
some signal for separating clearly low-rated from clearly high-rated films. Ablation and
masked-name experiments should be reported to show whether the signal is robust or partly
driven by artifacts such as names, genre vocabulary, and formatting.



In [14]:
# ============================================================
# FINAL STATS CELL: PRECISION / RECALL / CONFUSION MATRIX
# Does NOT rerun any training.
# Uses predictions_df if it exists, otherwise loads saved predictions CSV.
# ============================================================

import pandas as pd
import numpy as np

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    balanced_accuracy_score,
    roc_auc_score,
)

# Try to use in-memory predictions first.
# If the kernel was restarted, load the saved predictions file.
if "predictions_df" not in globals():
    possible_prediction_files = [
        "task_c_predictions.csv",
        "task_c_variant_predictions.csv",
        "task_c_cv_predictions.csv",
        "task_c_fast_predictions.csv",
        "task_c_threshold_predictions.csv",
        "task_c_word_char_predictions.csv",
        "task_c_model_predictions.csv",
    ]

    loaded = False
    for file_name in possible_prediction_files:
        try:
            predictions_df = pd.read_csv(file_name)
            print(f"Loaded predictions from: {file_name}")
            loaded = True
            break
        except FileNotFoundError:
            pass

    if not loaded:
        raise FileNotFoundError(
            "Could not find predictions_df in memory or a known saved predictions CSV. "
            "Check the filename printed by your training/evaluation cell."
        )
else:
    print("Using in-memory predictions_df")

required_cols = {"actual_label", "pred_label"}
missing = required_cols - set(predictions_df.columns)
if missing:
    raise ValueError(f"predictions_df is missing columns: {missing}")

y_true = predictions_df["actual_label"].astype(int).to_numpy()
y_pred = predictions_df["pred_label"].astype(int).to_numpy()

print("\nLabel meaning:")
print("0 = low / weak movie, IMDb <= low threshold")
print("1 = high / strong movie, IMDb >= high threshold")

# Confusion matrix: rows = actual, cols = predicted
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

confusion_df = pd.DataFrame(
    cm,
    index=["Actual Low (0)", "Actual High (1)"],
    columns=["Pred Low (0)", "Pred High (1)"]
)

print("\nConfusion matrix:")
display(confusion_df)

low_precision = tn / max(1, tn + fn)
low_recall = tn / max(1, tn + fp)

high_precision = tp / max(1, tp + fp)
high_recall = tp / max(1, tp + fn)

summary = {
    "accuracy": accuracy_score(y_true, y_pred),
    "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
    "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
    "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
    "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
    "low_precision": low_precision,
    "low_recall": low_recall,
    "low_f1": f1_score(y_true, y_pred, labels=[0], average="macro", zero_division=0),
    "high_precision": high_precision,
    "high_recall": high_recall,
    "high_f1": f1_score(y_true, y_pred, labels=[1], average="macro", zero_division=0),
    "n_total": len(y_true),
    "n_actual_low": int(np.sum(y_true == 0)),
    "n_actual_high": int(np.sum(y_true == 1)),
    "n_pred_low": int(np.sum(y_pred == 0)),
    "n_pred_high": int(np.sum(y_pred == 1)),
}

if "pred_prob_high" in predictions_df.columns:
    y_prob = predictions_df["pred_prob_high"].astype(float).to_numpy()
    if len(np.unique(y_true)) == 2:
        summary["roc_auc"] = roc_auc_score(y_true, y_prob)

summary_df = pd.DataFrame([summary]).T.reset_index()
summary_df.columns = ["metric", "value"]

print("\nOverall held-out stats from saved fold predictions:")
display(summary_df)

print("\nSklearn classification report:")
print(
    classification_report(
        y_true,
        y_pred,
        labels=[0, 1],
        target_names=["Low / weak", "High / strong"],
        zero_division=0,
    )
)

# Optional: fold-by-fold precision/recall if fold column exists
if "fold" in predictions_df.columns:
    fold_rows = []

    for fold, fold_df in predictions_df.groupby("fold"):
        yt = fold_df["actual_label"].astype(int).to_numpy()
        yp = fold_df["pred_label"].astype(int).to_numpy()

        fold_cm = confusion_matrix(yt, yp, labels=[0, 1])
        f_tn, f_fp, f_fn, f_tp = fold_cm.ravel()

        row = {
            "fold": fold,
            "accuracy": accuracy_score(yt, yp),
            "balanced_accuracy": balanced_accuracy_score(yt, yp),
            "low_precision": f_tn / max(1, f_tn + f_fn),
            "low_recall": f_tn / max(1, f_tn + f_fp),
            "high_precision": f_tp / max(1, f_tp + f_fp),
            "high_recall": f_tp / max(1, f_tp + f_fn),
            "macro_f1": f1_score(yt, yp, average="macro", zero_division=0),
            "n_test": len(yt),
        }

        if "pred_prob_high" in fold_df.columns and len(np.unique(yt)) == 2:
            row["roc_auc"] = roc_auc_score(yt, fold_df["pred_prob_high"].astype(float).to_numpy())

        fold_rows.append(row)

    fold_stats_df = pd.DataFrame(fold_rows)

    print("\nFold-by-fold precision/recall:")
    display(fold_stats_df)

    print("\nFold mean ± std:")
    display(
        fold_stats_df
        .drop(columns=["fold"])
        .agg(["mean", "std"])
        .T
        .reset_index()
        .rename(columns={"index": "metric"})
    )

Using in-memory predictions_df

Label meaning:
0 = low / weak movie, IMDb <= low threshold
1 = high / strong movie, IMDb >= high threshold

Confusion matrix:


,Pred Low (0),Pred High (1)
Actual Low (0),1397,529
Actual High (1),651,1161



Overall held-out stats from saved fold predictions:


,metric,value
0,accuracy,0.684323
1,balanced_accuracy,0.683033
2,macro_precision,0.684556
3,macro_recall,0.683033
4,macro_f1,0.683060
5,low_precision,0.682129
6,low_recall,0.725337
7,low_f1,0.703070
8,high_precision,0.686982
9,high_recall,0.640728



Sklearn classification report:
               precision    recall  f1-score   support

   Low / weak       0.68      0.73      0.70      1926
High / strong       0.69      0.64      0.66      1812

     accuracy                           0.68      3738
    macro avg       0.68      0.68      0.68      3738
 weighted avg       0.68      0.68      0.68      3738


Fold-by-fold precision/recall:


,fold,accuracy,balanced_accuracy,low_precision,low_recall,high_precision,high_recall,macro_f1,n_test,roc_auc
0,1,0.652000,0.651447,0.655696,0.674479,0.647887,0.628415,0.651479,750,0.737214
1,2,0.718667,0.717640,0.710462,0.760417,0.728614,0.674863,0.717650,750,0.777785
2,3,0.702667,0.700962,0.702179,0.743590,0.703264,0.658333,0.701174,750,0.736724
3,4,0.685484,0.684896,0.692308,0.703125,0.677966,0.666667,0.684972,744,0.721021
4,5,0.662634,0.659896,0.651481,0.744792,0.678689,0.575000,0.658787,744,0.717687



Fold mean ± std:


,metric,mean,std
0,accuracy,0.684290,0.027534
1,balanced_accuracy,0.682968,0.027638
2,low_precision,0.682425,0.027138
3,low_recall,0.725280,0.035433
4,high_precision,0.687284,0.030316
5,high_recall,0.640656,0.040681
6,macro_f1,0.682812,0.027904
7,n_test,747.600000,3.286335
8,roc_auc,0.738086,0.023906
